# Multi-column co-sorting quicksort — benchmark visualization

Renders the Google Benchmark sweep from `benchmark_multicolumn_gbench` across six dimensions.
The **algorithm** is the primary dimension:

- **algorithm**: `std_sort`, `2way_ins`, `2way_net`, `3way_ins`, `3way_net`
- **distribution**, **payload columns**, **data type** (u32/u64), **parallelism** (`fixed<N>` lanes), **working-set size** (L1 → 16×LLC)

`std_sort` is scalar (`lanes=na`); network variants exist for both u32 and u64.

> Each plot varies one dimension and fixes the rest. A plot can only show a dimension the
> JSON actually swept — if you filtered the benchmark to a single `cols`/`lanes`, that axis will be
> degenerate (the cells below print a warning when that happens). For the column/parallelism plots,
> run the benchmark without over-filtering those dimensions:

```bash
cmake --build build --target benchmark_multicolumn_gbench
taskset -c 0 ./build/benchmark_multicolumn_gbench \
    --benchmark_repetitions=5 --benchmark_display_aggregates_only=true \
    --benchmark_format=json --benchmark_out=build/mc_gbench.json
# (subset for speed but KEEP cols and lanes varying, e.g.:)
#   --benchmark_filter='dist=uniform_random/size=(L2|LLC)'
```

In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

JSON_PATH = 'build/mc_gbench_sample.json'   # <-- set to your --benchmark_out file
SIZE_ORDER = ['L1', 'L2', 'halfLLC', 'LLC', '2xLLC', '16xLLC']
ALGO_ORDER = ['std_sort', '2way_ins', '2way_net', '3way_ins', '3way_net']

In [ ]:
def load(path):
    raw = json.load(open(path))
    caches = {c['level']: c['size'] for c in raw['context'].get('caches', []) if c['type'] in ('Data', 'Unified')}
    rows = []
    for b in raw['benchmarks']:
        if b.get('run_type') == 'iteration' and b.get('repetitions', 1) not in (0, 1):
            continue
        if b.get('aggregate_name', '') not in ('', 'median'):
            continue
        if b.get('error_occurred') or 'count' not in b:
            continue
        dims = dict(kv.split('=') for kv in b['name'].split('/') if '=' in kv)
        count = int(b['count'])
        scale = {'ns': 1.0, 'us': 1e3, 'ms': 1e6, 's': 1e9}.get(b.get('time_unit', 'ns'), 1.0)
        rt_ns = float(b['real_time']) * scale
        rows.append(dict(algo=dims['algo'], dtype=dims['type'], lanes=int(b['lanes']), dist=dims['dist'],
                         cols=int(b['cols']), size=dims['size'], count=count,
                         ns_per_key=rt_ns / count if count else float('nan')))
    df = pd.DataFrame(rows)
    df['size'] = pd.Categorical(df['size'], categories=SIZE_ORDER, ordered=True)
    df['algo'] = pd.Categorical(df['algo'], categories=ALGO_ORDER, ordered=True)
    return df.sort_values(['dtype', 'algo', 'lanes', 'dist', 'cols', 'size']).reset_index(drop=True), caches

df, caches = load(JSON_PATH)
print('L1/L2/LLC bytes:', caches)
print('rows:', len(df), '| algos:', [a for a in ALGO_ORDER if a in set(df.algo)])
print('types:', sorted(df.dtype.unique()), '| lanes:', sorted(df.lanes.unique()))
print('dists:', sorted(df.dist.unique()), '| cols:', sorted(df.cols.unique()),
      '| sizes:', [s for s in SIZE_ORDER if s in set(df['size'].astype(str))])
df.head()

In [ ]:
# co-sort SIMD lanes present (excludes std_sort's lanes=0 sentinel)
def simd_lanes(df, dtype):
    return sorted(df[(df.dtype == dtype) & (df.algo != 'std_sort')].lanes.unique())

# rows for one (dtype, lanes, cols, size); std_sort is lanes-independent so include it always
def algo_slice(df, dtype, lanes, cols, size):
    m = (df.dtype == dtype) & (df.cols == cols) & (df['size'] == size)
    m &= (df.algo == 'std_sort') | (df.lanes == lanes)
    return df[m]

def warn_degenerate(values, axis):
    if len(set(values)) < 2:
        print(f'  [note] only one {axis} value present ({sorted(set(values))}) -> re-run the benchmark',
              f'without filtering {axis} to see it vary.')

## 1. Algorithm comparison (headline)

`ns/key` per algorithm, one panel per distribution, at fixed type / lanes / columns / size.

In [ ]:
def plot_algo_comparison(df, dtype, lanes, cols, size):
    dists = sorted(df.dist.unique())
    fig, axes = plt.subplots(1, len(dists), figsize=(3.2 * len(dists), 4), squeeze=False, sharey=True)
    for ax, dist in zip(axes[0], dists):
        s = algo_slice(df[df.dist == dist], dtype, lanes, cols, size).set_index('algo')
        s = s.reindex([a for a in ALGO_ORDER if a in set(s.index)])
        ax.bar(range(len(s)), s['ns_per_key'])
        ax.set_xticks(range(len(s))); ax.set_xticklabels(s.index, rotation=45, ha='right')
        ax.set_title(dist); ax.grid(alpha=0.3, axis='y')
    axes[0][0].set_ylabel('ns / key')
    fig.suptitle(f'{dtype}  lanes={lanes}  cols={cols}  size={size}'); fig.tight_layout(); return fig

for dtype in sorted(df.dtype.unique()):
    lanes = simd_lanes(df, dtype)[-1]
    biggest = [s for s in reversed(SIZE_ORDER) if s in set(df['size'].astype(str))][0]
    plot_algo_comparison(df, dtype, lanes, max(df.cols.unique()), biggest)
plt.show()

## 2. Cache-hierarchy sweep, by algorithm

`ns/key` as the working set grows, one line per algorithm, one panel per distribution.

In [ ]:
def plot_cache_sweep(df, dtype, lanes, cols):
    dists = sorted(df.dist.unique())
    fig, axes = plt.subplots(1, len(dists), figsize=(3.4 * len(dists), 4), squeeze=False, sharey=True)
    for ax, dist in zip(axes[0], dists):
        for algo in [a for a in ALGO_ORDER if a in set(df.algo)]:
            pts = []
            for size in SIZE_ORDER:
                s = algo_slice(df[(df.dist == dist) & (df.algo == algo)], dtype, lanes, cols, size)
                if not s.empty: pts.append((size, s['ns_per_key'].iloc[0]))
            if pts: ax.plot([p[0] for p in pts], [p[1] for p in pts], marker='o', label=algo)
        ax.set_title(dist); ax.set_xlabel('working set'); ax.tick_params(axis='x', rotation=45); ax.grid(alpha=0.3)
    axes[0][0].set_ylabel('ns / key'); axes[0][-1].legend(fontsize=8)
    fig.suptitle(f'{dtype}  lanes={lanes}  cols={cols}'); fig.tight_layout(); return fig

for dtype in sorted(df.dtype.unique()):
    plot_cache_sweep(df, dtype, simd_lanes(df, dtype)[-1], max(df.cols.unique()))
plt.show()

## 3. Parallelism scaling

`ns/key` vs `fixed<N>` lanes, one line per algorithm, one panel per distribution (fixed columns / size).
Only the partition is vectorized. `std_sort` is scalar → dashed reference line.
Needs the benchmark run with **multiple lane counts** (u32: 4/8/16, u64: 2/4/8).

In [ ]:
def plot_parallelism(df, dtype, cols, size):
    lanes_present = simd_lanes(df, dtype)
    warn_degenerate(lanes_present, 'lanes')
    dists = sorted(df.dist.unique())
    fig, axes = plt.subplots(1, len(dists), figsize=(3.4 * len(dists), 4), squeeze=False, sharey=True)
    for ax, dist in zip(axes[0], dists):
        base = df[(df.dtype == dtype) & (df.dist == dist) & (df.cols == cols) & (df['size'] == size)]
        for algo in [a for a in ALGO_ORDER if a != 'std_sort' and a in set(base.algo)]:
            s = base[base.algo == algo].sort_values('lanes')
            ax.plot(s['lanes'], s['ns_per_key'], marker='s', label=algo)
        ref = base[base.algo == 'std_sort']
        if not ref.empty: ax.axhline(ref['ns_per_key'].iloc[0], ls='--', color='gray', label='std_sort')
        ax.set_title(dist); ax.set_xlabel('lanes (fixed<N>)'); ax.set_xscale('log', base=2)
        ax.set_xticks(lanes_present); ax.set_xticklabels(lanes_present); ax.grid(alpha=0.3)
    axes[0][0].set_ylabel('ns / key'); axes[0][-1].legend(fontsize=8)
    fig.suptitle(f'{dtype}  cols={cols}  size={size}: parallelism'); fig.tight_layout(); return fig

for dtype in sorted(df.dtype.unique()):
    biggest = [s for s in reversed(SIZE_ORDER) if s in set(df['size'].astype(str))][0]
    size = 'LLC' if 'LLC' in set(df['size'].astype(str)) else biggest
    plot_parallelism(df, dtype, max(df.cols.unique()), size)
plt.show()

## 4. Payload-column scaling

`ns/key` vs number of co-sorted columns, one line per algorithm, one panel per distribution.
The slope is the marginal cost per carried column. Needs **multiple column counts** in the run.

In [ ]:
def plot_columns(df, dtype, lanes, size):
    cols_present = sorted(df.cols.unique())
    warn_degenerate(cols_present, 'cols')
    dists = sorted(df.dist.unique())
    fig, axes = plt.subplots(1, len(dists), figsize=(3.4 * len(dists), 4), squeeze=False, sharey=True)
    for ax, dist in zip(axes[0], dists):
        for algo in [a for a in ALGO_ORDER if a in set(df.algo)]:
            pts = []
            for cols in cols_present:
                s = algo_slice(df[(df.dist == dist) & (df.algo == algo)], dtype, lanes, cols, size)
                if not s.empty: pts.append((cols, s['ns_per_key'].iloc[0]))
            if pts: ax.plot([p[0] for p in pts], [p[1] for p in pts], marker='^', label=algo)
        ax.set_title(dist); ax.set_xlabel('payload columns'); ax.set_xticks(cols_present); ax.grid(alpha=0.3)
    axes[0][0].set_ylabel('ns / key'); axes[0][-1].legend(fontsize=8)
    fig.suptitle(f'{dtype}  lanes={lanes}  {size}: column scaling'); fig.tight_layout(); return fig

for dtype in sorted(df.dtype.unique()):
    biggest = [s for s in reversed(SIZE_ORDER) if s in set(df['size'].astype(str))][0]
    plot_columns(df, dtype, simd_lanes(df, dtype)[-1], biggest)
plt.show()

## 5. Summary table

Median `ns/key` by algorithm × distribution for a chosen type / lanes / columns / size.

In [ ]:
def summary(df, dtype, lanes, cols, size):
    return algo_slice(df, dtype, lanes, cols, size).pivot_table(index='algo', columns='dist', values='ns_per_key', observed=True).round(2)

biggest = [s for s in reversed(SIZE_ORDER) if s in set(df['size'].astype(str))][0]
summary(df, 'u32', simd_lanes(df, 'u32')[-1], max(df.cols.unique()), biggest)